In [5]:
import time, os
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [6]:
# FIPS code for each State and Territory to use in URL construction for downloading school data for each state
state_FIPS = {
    'Alabama': '01',
    'Alaska': '02',
    'Arizona': '04',
    'Arkansas': '05',
    'California': '06',
    'Colorado': '08',
    'Connecticut': '09',
    'Delaware': '10',
    'Florida': '12',
    'Georgia': '13',
    'Hawaii': '15',
    'Idaho': '16',
    'Illinois': '17',
    'Indiana': '18',
    'Iowa': '19',
    'Kansas': '20',
    'Kentucky': '21',
    'Louisiana': '22',
    'Maine': '23',
    'Maryland': '24',
    'Massachusetts': '25',
    'Michigan': '26',
    'Minnesota': '27',
    'Mississippi': '28',
    'Missouri': '29',
    'Montana': '30',
    'Nebraska': '31',
    'Nevada': '32',
    'New Hampshire': '33',
    'New Jersey': '34',
    'New Mexico': '35',
    'New York': '36',
    'North Carolina': '37',
    'North Dakota': '38',
    'Ohio': '39',
    'Oklahoma': '40',
    'Oregon': '41',
    'Pennsylvania': '42',
    'Rhode Island': '44',
    'South Carolina': '45',
    'South Dakota': '46',
    'Tennessee': '47',
    'Texas': '48',
    'Utah': '49',
    'Vermont': '50',
    'Virginia': '51',
    'Washington': '53',
    'West Virginia': '54',
    'Wisconsin': '55',
    'Wyoming': '56'
}

keep_cols = ['PSS_SCHOOL_ID', 'PSS_INST', 'LoGrade', 'HiGrade', 'PSS_ADDRESS', 'PSS_CITY', 'PSS_STABB', 'PSS_ZIP5', 'PSS_PHONE',
             'PSS_ENROLL_T', 'PSS_FTE_TEACH', 'PSS_RELIG', 'PSS_COMM_TYPE', 'PSS_COUNTY_NAME', 'PSS_ASSOC_1', 'PSS_ASSOC_2', 'PSS_ASSOC_3']

final_cols = ['NCES School ID', 'Account Name', 'Low Grade', 'High Grade', 'Billing Street', 'Billing City', 'Billing State', 'Billing ZIP',
              'Phone', 'Number of Students Served', 'Number of Teachers', 'School Type', 'School Environment', 'County Name']

# Set download directory
download_dir = 'C:\\Users\\mridd\\Desktop\\web_scraping_testing\\sch_data_download_testing\\private_sd_downloads'

# Configure Chrome options for automatic download
chrome_options = webdriver.ChromeOptions()
prefs = {"download.default_directory": download_dir} # Establish and add download directory preference
chrome_options.add_experimental_option("prefs", prefs)
chrome_options.add_argument("--headless=new")  # Run Chrome in headless mode (optional)

In [7]:
def create_sd_csv(excel_file, state):
    df = pd.concat(excel_file, ignore_index=True)

    # Change column names to values in keep_cols
    df.columns = df.loc[4]
    df = df[keep_cols]
    df = df.loc[5:].reset_index(drop=True) # Get only rows containing school data, reset index

    # Create masks to handle empty values for different columns
    num_students_empty_mask = (df['PSS_ENROLL_T'] == '–') | (df['PSS_ENROLL_T'] == '†') | (df['PSS_ENROLL_T'].isna())
    num_teachers_empty_mask = (df['PSS_FTE_TEACH'] == '–') | (df['PSS_FTE_TEACH'] == '†') | (df['PSS_FTE_TEACH'].isna())
    ungraded_grade_empty_mask = (df['LoGrade'] == '–') & (df['HiGrade'] == '–')

    # Assign empty string to empty values, convert non-empty values to appropriate data types
    df.loc[num_students_empty_mask, 'PSS_ENROLL_T'] = ''
    df.loc[~num_students_empty_mask, 'PSS_ENROLL_T'] = df.loc[~num_students_empty_mask, 'PSS_ENROLL_T'].astype(int)

    df.loc[num_teachers_empty_mask, 'PSS_FTE_TEACH'] = ''
    df.loc[~num_teachers_empty_mask, 'PSS_FTE_TEACH'] = df.loc[~num_teachers_empty_mask, 'PSS_FTE_TEACH'].astype(float)

    # Create Locale mappings/masks and update Locale column based on Locale Code values
    urban_mask = (df['PSS_COMM_TYPE'] == '1')
    suburban_mask = df['PSS_COMM_TYPE'].astype(int).isin(['2','3'])
    rural_mask = (df['PSS_COMM_TYPE'] == '4')

    df.loc[urban_mask, 'PSS_COMM_TYPE'] = 'Urban'
    df.loc[suburban_mask, 'PSS_COMM_TYPE'] = 'Suburban'
    df.loc[rural_mask, 'PSS_COMM_TYPE'] = 'Rural'

    # Calculating Low and High Grade Levels based on encoded values in columns
    grade_map = {'-1': '', '1': '', '2': 'PK', '3': 'KG', '4': 'KG', '5': 'KG'} # Map for ungraded schools
    # print(list(grade_map.keys()))

    low_grade_kg_and_lower_mask = (df['LoGrade'].isin(list(grade_map.keys())))
    high_grade_kg_and_lower_mask = (df['HiGrade'].isin(list(grade_map.keys())))

    df.loc[low_grade_kg_and_lower_mask, 'LoGrade'] = df.loc[low_grade_kg_and_lower_mask, 'LoGrade'].map(grade_map)
    df.loc[high_grade_kg_and_lower_mask, 'HiGrade'] = df.loc[high_grade_kg_and_lower_mask, 'HiGrade'].map(grade_map)

    df.loc[~low_grade_kg_and_lower_mask, 'LoGrade'] = (df.loc[~low_grade_kg_and_lower_mask, 'LoGrade'].astype(int) - 5).astype(str)
    df.loc[~high_grade_kg_and_lower_mask, 'HiGrade'] = (df.loc[~high_grade_kg_and_lower_mask, 'HiGrade'].astype(int) - 5).astype(str)

    # Fix phone number formatting
    df['PSS_PHONE'] = '(' + df['PSS_PHONE'].str.slice(0, 3) + ') ' + df['PSS_PHONE'].str.slice(3, 6) + '-' + df['PSS_PHONE'].str.slice(6)

    # Change state abbreviation to full name
    df['PSS_STABB'] = state

    # Determine school types based on Religious/Indepenent affiliate/association status
    religious_mask = (df['PSS_RELIG'].isin(['1','2']))
    independent_mask = (df['PSS_ASSOC_1'].str.lower().str.contains('independent')) | (df['PSS_ASSOC_2'].str.lower().str.contains('independent')) | (df['PSS_ASSOC_3'].str.lower().str.contains('independent'))

    df.loc[religious_mask, 'PSS_RELIG'] = 'Parochial'
    df.loc[independent_mask, 'PSS_RELIG'] = 'Independent'
    df.loc[~religious_mask & ~independent_mask, 'PSS_RELIG'] = 'Private'

    df.drop(columns=['PSS_ASSOC_1', 'PSS_ASSOC_2', 'PSS_ASSOC_3'], inplace=True) # Drop association columns since they won't be used in final version

    # Update columns to final version we want
    df.columns = final_cols

    # Add 'School' type label for each entry
    df['Type'] = 'School'

    # Temp output
    df.to_csv(f'{download_dir}\\{state}_sd.csv', index=False)

In [8]:
# Logic to navigate to School Data Excel file and download it
for state, fips in state_FIPS.items():

    # Create fresh Selenium driver instance
    driver = webdriver.Chrome(options=chrome_options)

    # Construct URL for each state using FIPS code
    url = f"""https://nces.ed.gov/surveys/pss/privateschoolsearch/school_list.asp?Search=1&SchoolName=&SchoolID=&Address=&City=&State={fips}&Zip=&Miles=&County=&PhoneAreaCode=&Phone=&Religion=&Association=&SchoolType=&Coed=&NumOfStudents=&NumOfStudentsRange=more&IncGrade=-1&LoGrade=-1&HiGrade=-1"""

    print(f'Downloading private school data for {state}...')
    try:
        driver.get(url) # Navigate to the specified URL
        wait = WebDriverWait(driver, 10) # Initialize WebDriverWait with a timeout of 10 seconds

        # print(f'Before clicking Excel link: {len(driver.window_handles)}') # For Debugging: Check number of windows before clicking Excel link

        # Click on link that opens window with Excel download link
        wait.until(EC.element_to_be_clickable((By.CLASS_NAME, 'excelclass'))).click()
        time.sleep(20)

        # print(f'After clicking Excel link: {len(driver.window_handles)}') # For Debugging: Check number of windows after clicking Excel link to ensure new window opened

        # Switch to the new window that opened after clicking the Excel link
        driver.switch_to.window(driver.window_handles[-1])

        wait.until(EC.element_to_be_clickable((By.LINK_TEXT, 'Download Excel File'))).click() # Click Excel download link in new window
        time.sleep(10) # Pause to allow time for download to occur

        # Rename the downloaded Excel file to test_download.csv
        files = os.listdir(download_dir)
        excel_files = [f for f in files if f.endswith('.xlsx') or f.endswith('.xls')]
        if excel_files:
            excel_file = pd.read_html(os.path.join(download_dir, excel_files[0])) # Read the downloaded Excel file into a DataFrame

            print(f'{state}: {excel_files[0]}')
            create_sd_csv(excel_file, state) # Create CSV file from the downloaded Excel file

            os.remove(os.path.join(download_dir, excel_files[0])) # Remove the original Excel file after conversion to CSV
    except Exception as e:
        print(f'Error downloading data for {state}: {e}')
    finally:
        print(f'{state} Download Complete\n')
        driver.quit() # Hard reset driver after each state

Alabama: ncesdata_860EC268.xls
Alabama Download Complete

Alaska: ncesdata_97D1A903.xls
Alaska Download Complete

Arizona: ncesdata_3BE6CE44.xls
Arizona Download Complete

Arkansas: ncesdata_28206B8F.xls
Arkansas Download Complete

California: ncesdata_ACA679E1.xls
California Download Complete

Colorado: ncesdata_9EF8A1DF.xls
Colorado Download Complete

Connecticut: ncesdata_1A91DD8.xls
Connecticut Download Complete

Delaware: ncesdata_5ED86A49.xls
Delaware Download Complete

Florida: ncesdata_5A0818D3.xls
Florida Download Complete

Georgia: ncesdata_217C6F3D.xls
Georgia Download Complete

Hawaii: ncesdata_D759AACD.xls
Hawaii Download Complete

Idaho: ncesdata_BCF02354.xls
Idaho Download Complete

Illinois: ncesdata_3B42D935.xls
Illinois Download Complete

Indiana: ncesdata_85C13403.xls
Indiana Download Complete

Iowa: ncesdata_F91A9C77.xls
Iowa Download Complete

Kansas: ncesdata_F82CE21A.xls
Kansas Download Complete

Kentucky: ncesdata_3BB770C0.xls
Kentucky Download Complete

Louisia

In [ ]:
# Create fresh Selenium driver instance
driver = webdriver.Chrome(options=chrome_options)
state = 'Alabama'

# Construct URL for each state using FIPS code
url = f"https://nces.ed.gov/surveys/pss/privateschoolsearch/school_list.asp?Search=1&SchoolName=&SchoolID=&Address=&City=&State=01&Zip=&Miles=&County=&PhoneAreaCode=&Phone=&Religion=&Association=&SchoolType=&Coed=&NumOfStudents=&NumOfStudentsRange=more&IncGrade=-1&LoGrade=-1&HiGrade=-1"

print(f'Downloading private school data for Alabama...')
try:
    driver.get(url) # Navigate to the specified URL
    wait = WebDriverWait(driver, 10) # Initialize WebDriverWait with a timeout of 10 seconds

    # Click on link that opens window with Excel download link
    wait.until(EC.element_to_be_clickable((By.CLASS_NAME, 'excelclass'))).click()
    time.sleep(20)

    # Switch to the new window that opened after clicking the Excel link
    driver.switch_to.window(driver.window_handles[-1])

    wait.until(EC.element_to_be_clickable((By.LINK_TEXT, 'Download Excel File'))).click() # Click Excel download link in new window
    time.sleep(5) # Pause to allow time for download to occur

    # Rename the downloaded Excel file to test_download.csv
    files = os.listdir(download_dir)
    excel_files = [f for f in files if f.endswith('.xlsx') or f.endswith('.xls')]
    if excel_files:
        excel_file = pd.read_html(os.path.join(download_dir, excel_files[0])) # Read the downloaded Excel file into a DataFrame

        print(f'{state}: {excel_files[0]}')
        create_sd_csv(excel_file, state)

        os.remove(os.path.join(download_dir, excel_files[0])) # Remove the original Excel file after conversion to CSV

finally:
    print(f'Alabama Download Complete\n')
    driver.quit() # Hard reset driver after each state

Alabama: ncesdata_860EC268.xls
Alabama Download Complete

